In [1]:
import numpy as np
from scipy import stats

np.random.seed(3)  # номер варіанта = 3 (Одеса)
daily_temps = np.random.normal(loc=23, scale=2.5, size=30)

daily_temps

array([27.47157118, 24.09127463, 23.24124367, 18.34126824, 22.30652949,
       22.11310255, 22.7931463 , 21.43249831, 22.89045458, 21.80695492,
       19.71533812, 25.21155595, 25.20329511, 27.27393266, 23.12508411,
       21.98830646, 21.63660013, 19.13380671, 25.45591859, 20.24733092,
       20.03738368, 22.48587525, 26.71537089, 23.59179067, 20.44053715,
       21.217517  , 24.56311242, 22.59871659, 21.07790912, 22.42492319])

In [2]:
n = len(daily_temps)
x_bar = daily_temps.mean()          # вибіркове середнє
s = daily_temps.std(ddof=1)         # вибіркове стандартне відхилення (ddof=1)

print(f"n = {n}")
print(f"x_bar = {x_bar:.4f} C")
print(f"s     = {s:.4f} C")
print(f"min = {daily_temps.min():.3f}, max = {daily_temps.max():.3f}")

n = 30
x_bar = 22.6877 C
s     = 2.3121 C
min = 18.341, max = 27.472


In [3]:
mu0 = 21.8                            # стара норма з таблиці для варіанта 3
alpha = 0.05
alternative = "two-sided"            # H1: mu != mu0

print(f"mu0 = {mu0} C, H1: mu != {mu0}, alpha = {alpha}")

mu0 = 21.8 C, H1: mu != 21.8, alpha = 0.05


In [4]:
se = s / np.sqrt(n)                        # стандартна похибка середнього (Лекція 12)
z = (x_bar - mu0) / se                     # z-статистика
p_value = 2 * (1 - stats.norm.cdf(abs(z)))  # двобічний p-value

print(f"se       = {se:.4f} C")
print(f"z        = {z:.3f}")
print(f"p_value  = {p_value:.4f}")

se       = 0.4221 C
z        = 2.103
p_value  = 0.0355


In [5]:
# для контролю: однобічні варіанти (НЕ для висновку — тип обрано у Завданні 2)
p_right = 1 - stats.norm.cdf(z)   # H1: mu > mu0 (потепління)
p_left  = stats.norm.cdf(z)       # H1: mu < mu0 (охолодження)

print(f"p (двобічний)  = {p_value:.4f}")
print(f"p (правый хвіст) = {p_right:.4f}")
print(f"p (лівий хвіст)  = {p_left:.4f}")

p (двобічний)  = 0.0355
p (правый хвіст) = 0.0177
p (лівий хвіст)  = 0.9823


In [6]:
reject = p_value < alpha
crit_z = stats.norm.ppf(1 - alpha/2)
print(f"critical |z| = {crit_z:.4f}")
print(f"|z| = {abs(z):.4f} -> {'відхиляємо H0' if reject else 'не відхиляємо H0'}")
print(f"p = {p_value:.4f} < alpha = {alpha}: {reject}")

critical |z| = 1.9600
|z| = 2.1030 -> відхиляємо H0
p = 0.0355 < alpha = 0.05: True


In [7]:
z_crit = 1.96
ci_low  = x_bar - z_crit * se
ci_high = x_bar + z_crit * se

print(f"95% CI для mu: ({ci_low:.4f}, {ci_high:.4f}) C")
print(f"mu0 = {mu0} C потрапляє в CI: {ci_low <= mu0 <= ci_high}")
print(f"підрахований p-value: {p_value:.4f}")

95% CI для mu: (21.8604, 23.5151) C
mu0 = 21.8 C потрапляє в CI: False
підрахований p-value: 0.0355


## Контрольні питання

### 1. Що означає отримане p-value і чому це не «ймовірність того, що μ0 правильна»?

p = 0.0355 означає: **якби H0 (μ = 21.8 °C) була істинна**, то у 3.55% гіпотетичних повторних вибірок n = 30 з того самого розподілу ми отримали б $\bar{x}$, що відрізняється від 21.8 °C **щонайменше** на спостерене $|z| = 2.103$ (тобто $\bar{x} \ge 22.69$ або $\bar{x} \le 20.91$ °C).

Це не ймовірність істинності H0. p-value обчислюється **за припущенням істинності H0** і відповідає на питання «наскільки незвичайним був би такий результат, якби H0 була правильною», а не «яка ймовірність, що H0 правильна». Плутанина виникає через те, що H0 тут (норма з довідника) не є випадковою подією з «ймовірністю 0.5» — вона або істинна, або ні; а p-value обмежене [0; 1] і при достатньо великій вибірці може бути близьким до нуля для цілком правильної H0, тоді як для абсолютно хибної H0 p все одно не дорівнює нулю. Правильна форма висновку: «дані суперечать H0 на рівні α = 0.05», а не «H0 істинна з ймовірністю 3.55%».

### 2. Чому тип тесту обирають до того, як побачено результат?

Тому що **напрямок H1 задає, які дані вже не мають права здивувати дослідника**. Якщо очікування зміни в конкретний бік не було сформульоване до збору даних, то, побачивши $\bar{x}$, дослідник може вибрати «зручний» хвіст, і тоді та сама інформація дасть удвічі менше p-value (0.0177 замість 0.0355 у нашому випадку). Це не збільшує інформації в даних — воно лише зменшує захист від випадкових відхилень: реальна ймовірність помилки I роду стає α = 0.025 (а при перегляді в обох напрямках — до 0.05), тобто формально заявлений рівень значущості більше не виконується. Отже, тип тесту обирають **апріорно**, згідно з науковим питанням, а не за результатом.

### 3. Помилки I і II роду в контексті саме наших даних

Див. розділ «Помилки I і II роду» вище: помилка I — оголосити потепління в Одесі, якого немає (хибне оновлення довідника на ≈ 0.9 °C); помилка II — лишити в довіднику норму 21.8 °C, попри те, що реальне липневе середнє вже близько 22.7 °C. Це різні помилки, бо перша стосується **хибно доданого ефекту** (небезпека для коректності знань), а друга — **пропущеного реального ефекту** (небезпека для практики).

### 4. Чи узгодилися висновки тесту й довірчого інтервалу?

**Так, і це не випадковість, а тотожність.** Двобічний тест відхиляє H0 ⇔ $|x̄ - \mu_0| > z_{1-α/2}\,se$ ⇔ $\mu_0 \notin [x̄ - z_{1-α/2} se,\, x̄ + z_{1-α/2} se]$. Тому при тому самому α і тому самому типі тесту обидва способи дають однаковий результат завжди: у нас p = 0.0355 < 0.05, і μ0 = 21.8 не потрапляє в (21.860; 23.515). Розбіжність могла б виникнути лише якщо б порівнювали різні рівні (напр. 90% CI з α = 0.05) або різні типи тесту (однобічній p проти двобічного CI).

In [ ]:
# Зведена таблиця результатів
print(f"Місто: Одеса (варіант 3)")
print(f"n = {n}, x_bar = {x_bar:.4f} C, s = {s:.4f} C")
print(f"mu0 = {mu0} C")
print(f"H0: mu = {mu0} / H1: mu != {mu0} (двобічний)")
print(f"se = {se:.4f}, z = {z:.3f}, p = {p_value:.4f}")
print(f"95% CI = ({ci_low:.4f}, {ci_high:.4f}) C; mu0 в CI: {ci_low <= mu0 <= ci_high}")
print(f"Рішення при alpha = {alpha}: {'відхилити H0' if reject else 'не відхиляти H0'}")